In [ ]:
import nltk
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from nltk.corpus import stopwords
from nltk.util import ngrams
from collections import Counter

In [2]:
data=pd.read_csv(r'csvs\Cleaned_constituency_data.csv')

In [13]:
# Step 3: Create Document-Term Matrix (DTM)
data['Extracted_Text'] = data['Extracted_Text'].fillna("")  # Replace NaN with an empty string
vectorizer = CountVectorizer(
    ngram_range=(1, 2),  # Reduce to bigrams max
    stop_words='english',
    max_df=0.8,          # Ignore terms in >80% of docs
    min_df=5,            # Ignore terms in <5 docs
    max_features=100000,   # Limit vocab size
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
)   
dtm = vectorizer.fit_transform(data['Extracted_Text'])

# Convert DTM to DataFrame
dtm_df = pd.DataFrame(dtm.toarray(), columns=vectorizer.get_feature_names_out(), index=data['Title'])
dtm_df.head()

,aanndd,aanndd ffiinnaanncciiaall,ab,ababa,ababa action,abate,abated,abenomics,ability,ability contribute,...,zero rate,zero sum,zhou,zhou xiaochuan,zimbabwe,zimbabwe global,zimbabwe international,zing,zone,zones
Title,,,,,,,,,,,,,,,,,,,,,
"IMFC Statement by Christine Lagarde, President of the ECB",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"IMFC Statement by HE Haitham Al Ghais, Secretary General, OPEC",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"IMFC Statement by Ayman Al-Sayari, Governor of the Saudi Central Bank (SAMA)",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"IMFC Statement by Antoine Armand, Minister of the Economy, Finance and Industry, France",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"IMFC Statement by Luis Caputo, Minister of Economy, Argentina",0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
# Step 5: Alternative LDA Topic Modeling using Sklearn
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda_topics = lda.fit_transform(dtm)

# Function to display top words per topic
def display_topics(model, feature_names, num_words=5):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-num_words - 1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("\nLDA Topics :")
display_topics(lda, vectorizer.get_feature_names_out(), num_words=5)


LDA Topics :
Topic 1: percent, gdp, year, inflation, rate
Topic 2: surveillance, eu, quota, reform, members
Topic 3: oil, demand, developing, year, trade
Topic 4: social, cent, labour, employment, development
Topic 5: climate, development, challenges, financing, pandemic


In [16]:
import pandas as pd
import plotly.express as px

# Define readable topic labels
topic_labels = {
    0: 'Macroeconomics (gdp, inflation, rate)',
    1: 'IMF Governance (surveillance, quota, reform)',
    2: 'Global Trade & Development (oil, trade, developing)',
    3: 'Social Policy (labour, employment, social)',
    4: 'Climate & Pandemic (climate, financing, challenges)'
}

# Create topic weights DataFrame
topic_weights = pd.DataFrame(lda_topics, columns=[topic_labels[i] for i in range(lda.n_components)])
topic_weights['Year'] = data['Year']

# Average topic weights per year
topic_year_avg = topic_weights.groupby('Year').mean().reset_index()

# Melt for long-format Plotly input
topic_long = topic_year_avg.melt(id_vars='Year', var_name='Topic', value_name='Average Weight')

# Plotly line chart
fig = px.line(
    topic_long,
    x='Year',
    y='Average Weight',
    color='Topic',
    title='LDA Topic Prevalence Over Time',
    markers=True
)

fig.update_layout(
    legend_title_text='LDA Topics',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()


In [25]:
print(topic_year_avg)

    Year  Macroeconomics (gdp, inflation, rate)  \
0   2004                               0.105541   
1   2005                               0.131455   
2   2006                               0.150343   
3   2007                               0.166937   
4   2008                               0.198924   
5   2009                               0.211650   
6   2010                               0.167958   
7   2011                               0.165833   
8   2012                               0.187018   
9   2013                               0.240884   
10  2014                               0.206519   
11  2015                               0.242506   
12  2016                               0.184504   
13  2017                               0.189520   
14  2018                               0.167427   
15  2019                               0.139086   
16  2020                               0.114382   
17  2021                               0.119788   
18  2022                       

# Maps

In [17]:
# Manual mapping: map regions/authorities to actual countries (ISO-3)
authority_to_countries = {
    'European Commission': ['FRA', 'DEU', 'ITA', 'ESP', 'NLD', 'BEL', 'SWE', 'AUT', 'FIN', 'DNK', 'GRC', 'IRL', 'PRT', 'CYP', 'HRV', 'CZE', 'EST', 'HUN', 'LTU', 'LVA', 'MLT', 'PL'],
    'European Central Bank': ['FRA', 'DEU', 'ITA', 'ESP', 'NLD'],
    'European Council': ['FRA', 'DEU', 'ITA', 'ESP', 'NLD', 'BEL', 'SWE'],
    'Council of Economic and Finance Ministers (EU)': ['FRA', 'DEU', 'ITA', 'ESP'],
    'European Union': ['FRA', 'DEU', 'ITA', 'ESP', 'NLD', 'BEL'],
    'World Bank': [],  # Skip global authorities unless you map them
    'IMF': [],
    'International Monetary Fund': [],
    'World Trade Organization': [],
    'International Labour Organization': [],
    'Financial Stability Board': [],
    'United Nations': [],
    'Organization of the Petroleum Exporting Countries': ['SAU', 'IRN', 'IRQ', 'KWT', 'VEN', 'NGA', 'AGO', 'DZA', 'GAB', 'ARE', 'ECU', 'LBY', 'CIV', 'GNQ', 'KON'],
    # Add more mappings
}

# Map single countries
country_to_iso3 = {
    'United States': 'USA', 'France': 'FRA', 'Germany': 'DEU',
    'United Kingdom': 'GBR', 'Brazil': 'BRA', 'China': 'CHN',
    'India': 'IND', 'Russia': 'RUS', 'Australia': 'AUS', 'Canada': 'CAN',
    'South Africa': 'ZAF', 'Saudi Arabia': 'SAU', 'Japan': 'JPN',
    'Argentina': 'ARG', 'Spain': 'ESP', 'Italy': 'ITA',
    'Sweden': 'SWE', 'Norway': 'NOR', 'Switzerland': 'CHE',
    'Netherlands': 'NLD', 'Belgium': 'BEL', 'Turkey': 'TUR',
    'Indonesia': 'IDN', 'Nigeria': 'NGA', 'Republic of Korea': 'KOR',
    'Singapore': 'SGP', 'Malaysia': 'MYS', 'Mexico': 'MEX',
    'Czech Republic': 'CZE', 'Hungary': 'HUN', 'Chile': 'CHL',
    'Thailand': 'THA', 'Ghana': 'GHA', 'Colombia': 'COL',
    'Cameroon': 'CMR', 'Djibouti': 'DJI', 'Peru': 'PER',
    'Denmark': 'DNK', 'Finland': 'FIN', 'Estonia': 'EST',
    'Lithuania': 'LTU', 'Burkina Faso': 'BFA', 'Austria': 'AUT',
    "Côte d'Ivoire": 'CIV', "Côte d'Ivoire (French)": 'CIV',
    'Canada (French)': 'CAN', 'Djibouti (French)': 'DJI',
    'Kingdom of the Netherlands': 'NLD',
    'Democratic Republic of the Congo': 'COD',
    # Add any others as needed
}

In [18]:
# Add topic scores to the original data
lda_df = pd.DataFrame(lda_topics, columns=[f"Topic_{i}" for i in range(lda.n_components)])
data_lda = pd.concat([data.reset_index(drop=True), lda_df], axis=1)

# Expand region mapping as before
rows = []
for _, row in data_lda.iterrows():
    authority = row['Region/Authority']
    topic_scores = {f"Topic_{i}": row[f"Topic_{i}"] for i in range(lda.n_components)}
    
    if authority in country_to_iso3:
        iso_list = [country_to_iso3[authority]]
    elif authority in authority_to_countries:
        iso_list = authority_to_countries[authority]
    else:
        continue

    for iso in iso_list:
        entry = {'iso_alpha': iso}
        entry.update(topic_scores)
        rows.append(entry)

lda_country_df = pd.DataFrame(rows)

# Compute mean topic scores per country
lda_country_mean = lda_country_df.groupby('iso_alpha').mean().reset_index()

# Determine dominant topic per country
lda_country_mean['Dominant_Topic'] = lda_country_mean[[f'Topic_{i}' for i in range(lda.n_components)]].idxmax(axis=1)


In [19]:
topic_labels = {
    'Topic_0': 'Macroeconomics',
    'Topic_1': 'IMF Governance',
    'Topic_2': 'Global Trade',
    'Topic_3': 'Social Policy',
    'Topic_4': 'Climate & Pandemic'
}

lda_country_mean['Dominant_Topic_Label'] = lda_country_mean['Dominant_Topic'].map(topic_labels)


In [20]:
import plotly.express as px

fig = px.choropleth(
    lda_country_mean,
    locations='iso_alpha',
    color='Dominant_Topic_Label',
    title='Dominant LDA Topic by Country',
    color_discrete_sequence=px.colors.qualitative.Set2,
    projection='equirectangular'
)

fig.update_layout(
    legend_title_text='Dominant Topic',
    margin=dict(t=50, l=0, r=0, b=0),
    geo=dict(showframe=False, showcoastlines=True)
)

fig.show()
